In [1]:
from db_connector import list_all_tables, load_table_to_df
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. 데이터 로드
all_tables = list_all_tables()
required_tables = ['user_profiles', 'meal_candidates', 'meal_candidate_items', 'foods', 'food_logs', 'recommendation_candidates']
all_dfs = {table: load_table_to_df(table) for table in all_tables if table in required_tables}
print("데이터 로드 완료!")

데이터 로드 완료!


In [2]:
# 2. MMR(Maximal Marginal Relevance) 페널티 계산 함수
def calculate_mmr_penalties(user_id, meal_candidates, meal_candidate_items, all_dfs, current_date=None):
    """
    과거 1주일치 food_logs를 조회하여 유저가 섭취했던 음식(food) 및 식단(candidate)을 카운트하고 페널티를 부과합니다.
    
    Returns:
        dict: { candidate_id : (candidate_penalty, food_penalty) }
    """
    food_logs = all_dfs.get('food_logs', pd.DataFrame())
    rec_cands = all_dfs.get('recommendation_candidates', pd.DataFrame())

    # 식단 목록 파싱 (dict list or dataframe 처리)
    if isinstance(meal_candidates, pd.DataFrame):
        meal_candidates = meal_candidates.to_dict('records')
        
    items_df = meal_candidate_items if isinstance(meal_candidate_items, pd.DataFrame) else pd.DataFrame(meal_candidate_items)

    if food_logs.empty:
        return {c['candidate_id']: (0.0, 0.0) for c in meal_candidates}

    # 1) 유저의 식사 로그 필터링
    user_logs = food_logs[food_logs['user_id'] == user_id].copy()
    user_logs['consumed_at'] = pd.to_datetime(user_logs['consumed_at'])

    if current_date is None:
        # 현재 날짜가 명시되지 않은 경우, 데이터상 가장 최근 로그 날짜를 기준으로 하거나 오늘 날짜 사용
        if not user_logs.empty:
            current_date = user_logs['consumed_at'].max()
        else:
            current_date = datetime.now()
    else:
        current_date = pd.to_datetime(current_date)

    # 2) 1주일(7일) 이내의 로그 필터링
    week_ago = current_date - timedelta(days=7)
    recent_logs = user_logs[user_logs['consumed_at'] >= week_ago]

    # 3) 최근 1주간 섭취한 개별 식품(food_id) 카운트
    food_counts = recent_logs['food_id'].value_counts().to_dict()

    # 4) 최근 1주간 섭취한 식단(candidate_id) 카운트
    # food_logs에는 recommendation_candidate_id가 있으므로, 이를 통해 실제 candidate_id를 역추적
    if not rec_cands.empty and 'recommendation_candidate_id' in recent_logs.columns:
        merged_logs = recent_logs.merge(
            rec_cands[['recommendation_candidate_id', 'candidate_id']], 
            on='recommendation_candidate_id', 
            how='inner'
        )
        candidate_counts = merged_logs['candidate_id'].value_counts().to_dict()
    else:
        candidate_counts = {}

    # 5) 페널티 부여 로직 (가중치는 튜닝을 위해 분리)
    CANDIDATE_PENALTY_WEIGHT = 0.15  # 식단(조합)을 통째로 다시 먹을 경우 부과되는 페널티
    FOOD_PENALTY_WEIGHT = 0.05       # 식단 내에 과거에 먹었던 개별 식품이 있을 경우 부과되는 페널티
    
    penalties = {}
    
    for cand in meal_candidates:
        c_id = cand['candidate_id']
        
        # A. 식단(Candidate) 통째 반복 페널티 계산
        cand_count = candidate_counts.get(c_id, 0)
        cand_penalty = cand_count * CANDIDATE_PENALTY_WEIGHT
        
        # B. 개별 식품(Food) 반복 페널티 합산
        c_items = items_df[items_df['candidate_id'] == c_id]
        food_penalty = 0.0
        for f_id in c_items['food_id']:
            f_count = food_counts.get(f_id, 0)
            food_penalty += f_count * FOOD_PENALTY_WEIGHT
            
        penalties[c_id] = (cand_penalty, food_penalty)
        
    return penalties

In [3]:
# 3. MMR 함수 테스트 실행
# 임의의 식단 후보 5개를 추출하여 유저 1번을 대상으로 테스트
sample_cands = all_dfs['meal_candidates'].head(5)
sample_items = all_dfs['meal_candidate_items']

penalties = calculate_mmr_penalties(user_id=1, meal_candidates=sample_cands, meal_candidate_items=sample_items, all_dfs=all_dfs)

print("[유저 1번 MMR 패널티 테스트 결과]")
print("Candidate ID : (Candidate Penalty, Food Penalty)")
for c_id, (cp, fp) in penalties.items():
    print(f"Candidate {c_id:<4} : ({cp:.2f}, {fp:.2f})")

[유저 1번 MMR 패널티 테스트 결과]
Candidate ID : (Candidate Penalty, Food Penalty)
Candidate 50000 : (0.00, 0.00)
Candidate 50001 : (0.00, 0.00)
Candidate 50002 : (0.00, 0.00)
Candidate 50003 : (0.15, 0.05)
Candidate 50004 : (0.30, 0.10)
